## Solution to the Optimization Problem

Given the opponent's simulated feasible squads $\mathbb{W}_o$ (from `copula_estimation.ipynb`),
solve the manager's own "beat the opponent" problem:

$$\max_{w\in\mathbb{W}} \text{Prob}\{w^T\delta > G^{(r')}(\mathbb{W}_o, \delta)\} \tag{Eq:maxProb}$$

where $\delta$ is the (uncertain) vector of player points, $G_o = w_o^T\delta$ is one
opponent squad's realized score, $G^{(r)}$ is the $r$-th order statistic of
$\{G_o\}_{o=1}^O$, $r' = O + 1 - r$, and $r$ trades off risk: 50% (beat the median
opponent squad) is risk-neutral, up to 90% (beat nearly all of them) is risk-seeking.

**Theorem** (mean-variance reduction): with $Y_w = w^T\delta - G^{(r')} \sim
N(\mu_{Y_w}, \sigma^2_{Y_w})$,

$$
\mu_{Y_w} = w^T\mu_\delta - \mu_{G^{(r')}}, \qquad
\sigma^2_{Y_w} = w^T\Sigma_\delta w + \sigma^2_{G^{(r')}} - 2w^T\sigma_{\delta,G^{(r')}}
$$

and the solution to Eq:maxProb is $w(\lambda) \in \arg\max_{w\in\mathbb{W}} \mu_{Y_w} \mp
\lambda\sigma^2_{Y_w}$ ($-\lambda$ if $\mu_{Y_w}\geq0$, $+\lambda$ otherwise) for some
$\lambda\geq0$, with $\mu_{G^{(r')}}, \sigma^2_{G^{(r')}}, \sigma_{\delta,G^{(r')}}$
estimated by Monte Carlo since they don't depend on $w$.

**This notebook's plan**, following the Lemma and Algorithm in the write-up:

1. Generate $\mathbb{W}_o$ (already done in `copula_estimation.ipynb` -- reload its
   pipeline here and rerun for one opponent/gameweek).
2. Build $\mu_\delta$, $\Sigma_\delta$ for the player-points vector $\delta$.
3. Monte Carlo: draw $\delta \sim N(\mu_\delta,\Sigma_\delta)$ many times; for each draw,
   score every $w_o\in\mathbb{W}_o$ and take the $r'$-th order statistic $G^{(r')}$;
   estimate $\mu_{G^{(r')}}, \sigma^2_{G^{(r')}}, \sigma_{\delta,G^{(r')}}$ from these paired
   samples.
4. Solve the mean-variance problem for a grid of $\lambda$ via a Gurobi MIQP over the
   real squad constraints (15 players, 2/5/5/3 by position, budget, transfer limit) --
   this _is_ $\mathbb{W}$, defined by linear/quadratic constraints rather than an
   explicit enumerated candidate list.
5. Pick $\lambda^* = \arg\max_\lambda \widehat{\text{Prob}}(Y_{w_\lambda} > 0)$ from the
   same Monte Carlo sample, and return $w^* = w_{\lambda^*}$.


### Setup

Reloads the position-pool / `alpha_manager` / vine-fitting machinery from
`copula_estimation.ipynb` (copied rather than `%run`, so this notebook doesn't also
re-trigger that notebook's expensive 33-gameweek bulk loop as a side effect) so
`run_copula_pipeline` is available to regenerate $\mathbb{W}_o$ for a chosen
opponent/gameweek with current code.


In [5]:
import ast

import numpy as np
import pandas as pd
import joblib
import pyvinecopulib as pv
from scipy.stats import kendalltau
import gurobipy as gp
from gurobipy import GRB

player_data = pd.read_csv('../../rolled_data_24_25.csv')
league_selections = pd.read_csv('../../league_selections_df.csv')

LEAGUE_FITS = {
    'Goalkeeper': joblib.load('../estimates/league_models_gk'),
    'Defender': joblib.load('../estimates/league_models_def'),
    'Midfielder': joblib.load('../estimates/league_models_mid'),
    'Forward': joblib.load('../estimates/league_models_fwd'),
}

POSITION_BINARY_COL = {
    'Goalkeeper': 'goalkeeper_binary',
    'Defender': 'defender_binary',
    'Midfielder': 'midfielder_binary',
    'Forward': 'forward_binary',
}

SQUAD_SLOTS = {'Goalkeeper': 2, 'Defender': 5, 'Midfielder': 5, 'Forward': 3}
POSITIONS = ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']
BICOP_FAMILY_SET = [pv.BicopFamily.indep, pv.BicopFamily.gaussian, pv.BicopFamily.clayton,
                     pv.BicopFamily.gumbel, pv.BicopFamily.frank, pv.BicopFamily.joe]
TRUNC_LVL = 15


def get_league_fit_for_round(league_fits, target_round):
    available = [gw for gw in league_fits if gw <= target_round]
    if not available:
        raise ValueError(f"No league fit available at or before round {target_round}")
    return league_fits[max(available)]


def _position_pool(player_data, position, rnd, feats):
    pool = player_data[(player_data['round'] == rnd) & (player_data['position'] == position)] \
        .sort_values('element')
    elements = pool['element'].to_numpy()
    X_raw = pool[feats].fillna(0).to_numpy(dtype=float)
    return elements, X_raw


def alpha_manager(manager_fit, league_fit, X_new_raw, y_prev):
    X_std_new = (X_new_raw - league_fit["X_mean"]) / league_fit["X_std"]
    beta_m = league_fit["beta_mean"] + manager_fit["delta_beta_mean"]
    linear = X_std_new @ beta_m + manager_fit["gamma_mean"] * y_prev
    linear = np.clip(linear, -30, 30)
    return np.exp(linear)


def marginal_inclusion_probabilities(weights, k, rng, n_sim=3000):
    weights = np.clip(np.asarray(weights, dtype=float), 1e-12, None)
    n = len(weights)
    U = rng.uniform(size=(n_sim, n))
    keys = U ** (1.0 / weights)[None, :]
    topk_idx = np.argpartition(-keys, kth=k - 1, axis=1)[:, :k]
    counts = np.bincount(topk_idx.ravel(), minlength=n)
    return counts / n_sim


def detect_squad_overhaul(team_id, gw, league_selections, threshold=8):
    prev_rows = league_selections[(league_selections['team_id'] == team_id) & (league_selections['round'] == gw - 1)]
    cur_rows = league_selections[(league_selections['team_id'] == team_id) & (league_selections['round'] == gw)]
    if len(prev_rows) == 0 or len(cur_rows) == 0:
        return None, None
    prev_squad = set(ast.literal_eval(prev_rows.iloc[0]['squad']))
    cur_squad = set(ast.literal_eval(cur_rows.iloc[0]['squad']))
    n_changed = len(cur_squad - prev_squad)
    return n_changed, n_changed > threshold


def compute_free_transfers_available(team_id, target_round, league_selections, cap=5, overhaul_threshold=8):
    ft_bank = 1
    for w in range(2, target_round):
        n_changed, is_overhaul = detect_squad_overhaul(team_id, w, league_selections, threshold=overhaul_threshold)
        if n_changed is None:
            continue
        if is_overhaul:
            ft_bank = min(cap, ft_bank + 1)
        elif n_changed <= ft_bank:
            ft_bank = min(cap, (ft_bank - n_changed) + 1)
        else:
            ft_bank = 1
    return ft_bank


def get_owned(team_id, position, rnd, elements):
    rows = league_selections[(league_selections['team_id'] == team_id) & (league_selections['round'] == rnd)]
    if len(rows) == 0:
        return None
    y = np.asarray(ast.literal_eval(rows.iloc[0][POSITION_BINARY_COL[position]]), dtype=float)
    if len(y) != len(elements):
        return None
    return dict(zip(elements.tolist(), y.tolist()))


def position_alpha_at_week(team_id, position, manager_fit, league_fit, rnd):
    elements, X_raw = _position_pool(player_data, position, rnd, league_fit['feats'])
    prev_elements, _ = _position_pool(player_data, position, rnd - 1, league_fit['feats'])
    owned_prev_map = get_owned(team_id, position, rnd - 1, prev_elements) or {}
    y_prev = np.array([owned_prev_map.get(e, 0.0) for e in elements])
    alpha = alpha_manager(manager_fit, league_fit, X_raw, y_prev)
    owned_map = get_owned(team_id, position, rnd, elements)
    return dict(zip(elements.tolist(), alpha.tolist())), owned_map


def position_marginals_at_week(team_id, position, manager_fit, league_fit, rnd, rng):
    alpha_map, owned_map = position_alpha_at_week(team_id, position, manager_fit, league_fit, rnd)
    elements = np.array(list(alpha_map.keys()))
    alpha = np.array(list(alpha_map.values()))
    p = marginal_inclusion_probabilities(alpha, SQUAD_SLOTS[position], rng)
    return dict(zip(elements.tolist(), p.tolist())), owned_map


def latent_uniforms(X_hist, P_hist, rng):
    a = np.where(X_hist.values == 0, 0.0, 1.0 - P_hist.values)
    b = np.where(X_hist.values == 0, 1.0 - P_hist.values, 1.0)
    draws = rng.uniform(size=X_hist.shape)
    U = a + draws * (b - a)
    return pd.DataFrame(U, index=X_hist.index, columns=X_hist.columns)


def run_copula_pipeline(GW, TEAM_ID, N_TOP=30, N_SAMPLES=20_000, T_LIMIT=None, min_history_weeks=1,
                         seed=42, verbose=True):
    """Verbatim from copula_estimation.ipynb -- see that notebook for the full
    derivation/markdown of each step."""
    if T_LIMIT is None:
        T_LIMIT = compute_free_transfers_available(TEAM_ID, GW, league_selections)

    rng = np.random.default_rng(seed)
    manager_fits = joblib.load(f'./estimates/manager_fits_{GW}.joblib')[GW]

    pool_records = []
    for position in POSITIONS:
        manager_fit = manager_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        p_map, owned_map = position_marginals_at_week(TEAM_ID, position, manager_fit, league_fit, GW, rng)
        ranked = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)
        top_elements = {e for e, _ in ranked[:N_TOP]}
        owned_elements = {e for e, v in (owned_map or {}).items() if v == 1}
        for e in top_elements | owned_elements:
            pool_records.append({"element": e, "position": position})
    pool_df = pd.DataFrame(pool_records).sort_values(['position', 'element']).reset_index(drop=True)
    elements_ordered = pool_df['element'].tolist()
    elements_arr = np.array(elements_ordered)
    position_arr = pool_df['position'].to_numpy()
    M = len(elements_ordered)

    all_history_rounds = sorted(
        league_selections.loc[
            (league_selections['team_id'] == TEAM_ID) & (league_selections['round'] < GW) & (league_selections['round'] >= 4),
            'round'
        ].unique().tolist()
    )
    overhaul_rounds = [r for r in all_history_rounds if detect_squad_overhaul(TEAM_ID, r, league_selections)[1]]
    history_rounds = [r for r in all_history_rounds if r not in overhaul_rounds]
    if len(history_rounds) < min_history_weeks:
        if verbose:
            print(f"GW{GW} team {TEAM_ID}: only {len(history_rounds)} non-overhaul history weeks "
                  f"(< {min_history_weeks}), skipping")
        return None

    X_hist = pd.DataFrame(index=history_rounds, columns=elements_ordered, dtype=float)
    P_hist = pd.DataFrame(index=history_rounds, columns=elements_ordered, dtype=float)
    for position in POSITIONS:
        manager_fit = manager_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        pos_elements = pool_df.loc[pool_df['position'] == position, 'element'].tolist()
        for rnd in history_rounds:
            p_map, owned_map = position_marginals_at_week(TEAM_ID, position, manager_fit, league_fit, rnd, rng)
            for e in pos_elements:
                X_hist.loc[rnd, e] = (owned_map or {}).get(e, 0.0)
                P_hist.loc[rnd, e] = p_map.get(e, 1e-6)
    U_vals = latent_uniforms(X_hist, P_hist, rng).values

    tau_matrix = np.zeros((M, M))
    for i in range(M):
        for j in range(i + 1, M):
            tau, _ = kendalltau(U_vals[:, i], U_vals[:, j])
            tau = 0.0 if np.isnan(tau) else tau
            tau_matrix[i, j] = tau_matrix[j, i] = tau
    hub_scores = np.abs(tau_matrix).sum(axis=1)
    cvine_order = (np.argsort(-hub_scores) + 1).tolist()

    cvine_structure = pv.CVineStructure(order=cvine_order, trunc_lvl=TRUNC_LVL)
    cvine_controls = pv.FitControlsVinecop(family_set=BICOP_FAMILY_SET, selection_criterion="aic", trunc_lvl=TRUNC_LVL)
    vine = pv.Vinecop.from_data(U_vals, structure=cvine_structure, controls=cvine_controls)
    aic = vine.aic(U_vals)

    alpha_target, current_owned = {}, {}
    for position in POSITIONS:
        manager_fit = manager_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        alpha_map, owned_map = position_alpha_at_week(TEAM_ID, position, manager_fit, league_fit, GW)
        for e in pool_df.loc[pool_df['position'] == position, 'element']:
            alpha_target[e] = alpha_map.get(e, 1e-12)
            current_owned[e] = (owned_map or {}).get(e, 0.0)
    alpha_vec = np.array([alpha_target[e] for e in elements_ordered])
    current_vec = np.array([current_owned[e] for e in elements_ordered])
    value_map = player_data.loc[player_data['round'] == GW].set_index('element')['value'].to_dict()
    value_vec = np.array([value_map.get(e, np.nan) for e in elements_ordered])
    current_squad_value = float(current_vec @ value_vec)

    sim_U = vine.simulate(n=N_SAMPLES, seeds=[seed])
    w_star = np.zeros_like(sim_U, dtype=int)
    for position, k in SQUAD_SLOTS.items():
        idxs = np.where(position_arr == position)[0]
        w = np.clip(alpha_vec[idxs], 1e-12, None)
        keys = sim_U[:, idxs] ** (1.0 / w)[None, :]
        topk = np.argpartition(-keys, kth=k - 1, axis=1)[:, :k]
        rows = np.repeat(np.arange(sim_U.shape[0]), k)
        cols = idxs[topk.ravel()]
        w_star[rows, cols] = 1

    transfers = 0.5 * np.abs(w_star - current_vec[None, :]).sum(axis=1)
    squad_values = w_star @ value_vec
    transfer_ok = transfers <= T_LIMIT
    budget_ok = squad_values <= current_squad_value
    feasible_mask = transfer_ok & budget_ok

    feasible_squads = [tuple(elements_arr[row.astype(bool)].tolist()) for row in w_star[feasible_mask]]
    squad_counts = {}
    for sq in feasible_squads:
        squad_counts[sq] = squad_counts.get(sq, 0) + 1

    result = {
        "GW": GW, "team_id": TEAM_ID, "M": M, "elements_ordered": elements_ordered,
        "n_history_weeks": len(history_rounds), "overhaul_rounds": overhaul_rounds,
        "vine_aic": aic, "current_squad_value": current_squad_value,
        "n_samples": N_SAMPLES, "T_LIMIT": T_LIMIT,
        "n_feasible": int(feasible_mask.sum()), "n_unique_feasible_squads": len(squad_counts),
        "squad_counts": squad_counts,
    }
    if verbose:
        print(f"GW{GW:>2} team {TEAM_ID:>8}: M={M:>3}, history={len(history_rounds):>2}wks "
              f"({len(overhaul_rounds)} overhaul), T'={T_LIMIT}, AIC={aic:>9.1f}, "
              f"feasible={result['n_feasible']:>5}/{N_SAMPLES} ({result['n_unique_feasible_squads']} unique)")
    return result

### Parameters

`OWN_TEAM_ID` is manager 205 -- "us" throughout this whole project (the round-robin
fixture in `longitudinal.ipynb` always computes `opponents_by_round` _for_ manager 205).
`OPPONENT_TEAM_ID`/`GW` continue the running example from `copula_estimation.ipynb`
(GW38's opponent, team 13603, verified well-calibrated and with the longest available
history). `R_PCT` is the risk level $r$ (fraction of opponent squads to beat) --
50% is risk-neutral per the write-up.


In [6]:
OWN_TEAM_ID = 205
OPPONENT_TEAM_ID = 13603
GW = 38
R_PCT = 0.50  # risk-neutral: beat the median of the opponent's feasible squads

print(f"Manager {OWN_TEAM_ID} vs opponent {OPPONENT_TEAM_ID} at GW{GW}, r={R_PCT:.0%}")

Manager 205 vs opponent 13603 at GW38, r=50%


### Step 1: the opponent's feasible squads $\mathbb{W}_o$

Rerun `run_copula_pipeline` for the opponent -- this is exactly `copula_estimation.ipynb`'s
output, regenerated here with current code (dynamic $T'$, budget constraint, per-position
top-k sampling) rather than reloading a possibly-stale saved result.


In [7]:
opp_result = run_copula_pipeline(GW, OPPONENT_TEAM_ID, N_SAMPLES=100_000, seed=42)

opp_elements = opp_result['elements_ordered']
opp_squad_counts = opp_result['squad_counts']  # {squad_tuple: count}
O = len(opp_squad_counts)
print(f"\n{O} unique feasible opponent squads (Wo) to score against")

GW38 team    13603: M=120, history=30wks (4 overhaul), T'=3, AIC=   -907.0, feasible= 7573/100000 (7246 unique)

7246 unique feasible opponent squads (Wo) to score against


### Step 2: player-points vector $\delta$ -- $\mu_\delta$ and $\Sigma_\delta$

$\mu_\delta$ comes from `FPL Data/predictors`' walk-forward-validated `xP` regressor
(the most recently built, most rigorously backtested points model in the repo -- see
its own notebooks for the model zoo/feature-selection details): each player's predicted
`xP` for gameweek `GW` itself.

$\Sigma_\delta$ is _not_ the covariance of raw historical points (that would mostly
reflect "some players are nailed starters and some aren't", which $\mu_\delta$ already
captures) -- it's the empirical covariance of the model's own **residuals**
($xP_{\text{actual}} - xP_{\text{pred}}$) across all rounds before `GW`, i.e. the
genuine _prediction uncertainty_ structure (including how errors co-move across
players, e.g. two players on the same team missing a game together).

With ~780 players and lots of missing rounds (transfers, injuries, rotation), the raw
pairwise-deletion covariance is badly indefinite (over half its eigenvalues negative,
checked directly). Fixed via the standard nearest-PSD correction on the _correlation_
matrix (eigenvalue-clip then renormalize to unit diagonal, then rescale by the original
variances) rather than on the raw covariance directly, which preserves each player's
own variance exactly instead of distorting it.


In [8]:
def build_mu_sigma_delta(GW, predictors_dir='../../predictors', min_periods=3, min_eig_frac=1e-4):
    frames = []
    for pos in ['gk', 'def', 'mid', 'fwd']:
        df = pd.read_csv(f'{predictors_dir}/hist/{pos}_preds.csv')
        frames.append(df)
    all_preds = pd.concat(frames, ignore_index=True)

    mu_row = all_preds[all_preds['round'] == GW].drop_duplicates('element')
    hist = all_preds[all_preds['round'] < GW].copy()
    hist['resid'] = hist['xP_actual'] - hist['xP_pred']
    pivot = hist.pivot_table(index='round', columns='element', values='resid')

    elements = sorted(set(mu_row['element']) & set(pivot.columns))
    mu_delta = mu_row.set_index('element').loc[elements, 'xP_pred'].to_numpy(dtype=float)

    cov = pivot[elements].cov(min_periods=min_periods).fillna(0.0).to_numpy()
    cov = (cov + cov.T) / 2
    np.fill_diagonal(cov, np.maximum(np.diag(cov), 1e-4))

    std = np.sqrt(np.diag(cov))
    corr = cov / np.outer(std, std)
    np.fill_diagonal(corr, 1.0)

    eigvals, eigvecs = np.linalg.eigh(corr)
    floor = max(min_eig_frac * eigvals.max(), 1e-8)
    corr_psd = eigvecs @ np.diag(np.clip(eigvals, floor, None)) @ eigvecs.T
    d = np.sqrt(np.diag(corr_psd))
    corr_psd = corr_psd / np.outer(d, d)
    np.fill_diagonal(corr_psd, 1.0)
    corr_psd = (corr_psd + corr_psd.T) / 2

    sigma_delta = corr_psd * np.outer(std, std)
    return elements, mu_delta, sigma_delta


delta_elements, mu_delta, Sigma_delta = build_mu_sigma_delta(GW)
print(f"delta dimension P = {len(delta_elements)}")
print(f"min eigenvalue of Sigma_delta: {np.linalg.eigvalsh(Sigma_delta).min():.2e} (should be >= 0)")
print(f"mu_delta range: [{mu_delta.min():.2f}, {mu_delta.max():.2f}], mean {mu_delta.mean():.2f}")

delta dimension P = 781
min eigenvalue of Sigma_delta: 4.50e-05 (should be >= 0)
mu_delta range: [-0.79, 7.58], mean 1.04


### Step 3: own squad, budget, and player universe

Manager 205's _current_ squad is their GW37 squad (before GW38's decision); their
budget is that squad's value at GW38 prices (same convention as the opponent's budget
constraint in `copula_estimation.ipynb`); their transfer limit $T'_{\text{own}}$ is
their own banked free transfers via `compute_free_transfers_available`.

The player universe for the optimizer is every player with both a $\mu_\delta$/
$\Sigma_\delta$ entry (Step 2) _and_ a valid position/price at `GW` -- the ILP's own
constraints (budget, 2/5/5/3, transfer limit) define $\mathbb{W}$ directly, so there's
no need for the top-N pool truncation used for the opponent's copula stage.


In [9]:
own_row_prev = league_selections[
    (league_selections['team_id'] == OWN_TEAM_ID) & (league_selections['round'] == GW - 1)
].iloc[0]
own_current_squad = set(ast.literal_eval(own_row_prev['squad']))

gw_players = player_data[player_data['round'] == GW].drop_duplicates('element').set_index('element')
universe = sorted(set(delta_elements) & set(gw_players.index))
print(f"player universe: {len(universe)} players (out of {len(delta_elements)} with mu/Sigma, "
      f"{len(gw_players)} with GW{GW} price/position)")

own_budget = float(gw_players.loc[list(own_current_squad & set(gw_players.index)), 'value'].sum())
own_T_limit = compute_free_transfers_available(OWN_TEAM_ID, GW, league_selections)
print(f"own budget (GW37 squad value at GW{GW} prices): {own_budget}")
print(f"own free transfers banked entering GW{GW}: {own_T_limit}")

price_vec = gw_players.loc[universe, 'value'].to_numpy(dtype=float)
position_vec = gw_players.loc[universe, 'position'].to_numpy()
owned_vec = np.array([1.0 if e in own_current_squad else 0.0 for e in universe])

# re-index mu_delta/Sigma_delta onto `universe`'s order
delta_index = {e: i for i, e in enumerate(delta_elements)}
u_idx = np.array([delta_index[e] for e in universe])
mu_u = mu_delta[u_idx]
Sigma_u = Sigma_delta[np.ix_(u_idx, u_idx)]
P = len(universe)

player universe: 781 players (out of 781 with mu/Sigma, 784 with GW38 price/position)
own budget (GW37 squad value at GW38 prices): 1050.0
own free transfers banked entering GW38: 4


### Step 4: Monte Carlo samples of $(\delta, G^{(r')})$

Per the Lemma: draw $\delta\sim N(\mu_\delta,\Sigma_\delta)$; for _that same_ draw, score
every $w_o\in\mathbb{W}_o$ (weighted by how often the copula/vine simulation produced
it -- `opp_squad_counts`); take the $r'$-th order statistic across the opponent's
squads. Repeating this gives paired samples of $(\delta, G^{(r')})$, from which
$\mu_{G^{(r')}}$, $\sigma^2_{G^{(r')}}$, and $\sigma_{\delta,G^{(r')}}$ (the $P$-vector of
$\text{Cov}(\delta_p, G^{(r')})$) are estimated empirically -- none of them depend on our
own decision $w$.


In [10]:
N_MC = 3000
rng = np.random.default_rng(0)

# Wo as a (O, P) binary matrix in `universe`'s column order, weighted by squad_counts
opp_element_index = {e: i for i, e in enumerate(universe)}
Wo_rows, Wo_weights = [], []
for squad, count in opp_squad_counts.items():
    row = np.zeros(P)
    valid = True
    for e in squad:
        idx = opp_element_index.get(e)
        if idx is None:
            valid = False
            break
        row[idx] = 1.0
    if valid:
        Wo_rows.append(row)
        Wo_weights.append(count)
Wo = np.array(Wo_rows)
Wo_weights = np.array(Wo_weights, dtype=float)
Wo_weights /= Wo_weights.sum()
print(f"Wo matrix: {Wo.shape} ({Wo.shape[0]} / {O} opponent squads had all their players in `universe`)")

r_index_from_top = int(np.ceil((1 - R_PCT) * Wo.shape[0]))  # r' = O + 1 - r, 1-indexed from the top
r_index_from_top = min(max(r_index_from_top, 1), Wo.shape[0])

delta_samples = rng.multivariate_normal(mu_u, Sigma_u, size=N_MC, method='eigh')
G_all = delta_samples @ Wo.T  # (N_MC, O_valid): each opponent squad's score per MC draw
G_sorted = np.sort(G_all, axis=1)
G_r = G_sorted[:, -r_index_from_top]  # r'-th order statistic (from the top) per draw

mu_G_r = G_r.mean()
var_G_r = G_r.var(ddof=1)
sigma_delta_G_r = ((delta_samples - mu_u) * (G_r - mu_G_r)[:, None]).mean(axis=0)

print(f"mu_G_r' = {mu_G_r:.3f}, sigma^2_G_r' = {var_G_r:.3f}")
print(f"sigma_delta_G_r' range: [{sigma_delta_G_r.min():.3f}, {sigma_delta_G_r.max():.3f}]")

Wo matrix: (7246, 781) (7246 / 7246 opponent squads had all their players in `universe`)
mu_G_r' = 58.710, sigma^2_G_r' = 167.417
sigma_delta_G_r' range: [-26.820, 31.243]


### Step 5: mean-variance MIQP over a grid of $\lambda$

$\mathbb{W}$ is defined directly by the squad constraints (exactly 15 players, 2 GK/5
DEF/5 MID/3 FWD, budget $\leq$ `own_budget`, at most `own_T_limit` new players vs. the
current squad) rather than an enumerated candidate list -- solved as a Gurobi MIQP.
$\mu_{G^{(r')}}$ and $\sigma^2_{G^{(r')}}$ don't depend on $w$, so they drop out of the
$\arg\max$; only $w^T\mu_\delta - \lambda\big(w^T\Sigma_\delta w - 2w^T\sigma_{\delta,G^{(r')}}\big)$
matters for solving $w(\lambda)$ (the sign convention follows the Theorem, and in
practice $\mu_{Y_w}\geq0$ is the relevant branch once $w$ is any reasonably strong
squad).


In [11]:
def build_mean_variance_model(mu_u, Sigma_u, sigma_delta_G_r, price_vec, position_vec, owned_vec,
                               budget, T_limit):
    """
    Builds the Gurobi model, constraints, and the (lambda-independent) linear/quadratic
    expressions once. Building the dense quadratic expression over ~780 players is the
    expensive part (~10s) -- reusing this model and just swapping the objective per
    lambda (see solve_for_lambda) avoids paying that cost once per grid point.
    """
    P = len(mu_u)
    m = gp.Model('mean_variance')
    m.Params.OutputFlag = 0
    w = m.addVars(P, vtype=GRB.BINARY, name='w')

    m.addConstr(gp.quicksum(w[i] for i in range(P)) == 15)
    for pos, k in SQUAD_SLOTS.items():
        idxs = np.where(position_vec == pos)[0]
        m.addConstr(gp.quicksum(w[i] for i in idxs) == k)
    m.addConstr(gp.quicksum(price_vec[i] * w[i] for i in range(P)) <= budget)
    m.addConstr(gp.quicksum(w[i] * (1 - owned_vec[i]) for i in range(P)) <= T_limit)

    lin = gp.quicksum(mu_u[i] * w[i] for i in range(P))
    quad_var = gp.quicksum(Sigma_u[i, j] * w[i] * w[j] for i in range(P) for j in range(P) if Sigma_u[i, j] != 0)
    lin_cov = gp.quicksum(sigma_delta_G_r[i] * w[i] for i in range(P))
    return m, w, lin, quad_var, lin_cov


def solve_for_lambda(m, w, lin, quad_var, lin_cov, P, lam, sense=1):
    """sense=+1 for the mu_Yw >= 0 branch (maximize mean - lam*var), -1 otherwise
    (maximize mean + lam*var). Returns the optimal binary allocation w (len P) or None
    if infeasible. Reuses the model/constraints from build_mean_variance_model."""
    m.setObjective(lin - sense * lam * (quad_var - 2 * lin_cov), GRB.MAXIMIZE)
    m.optimize()
    if m.status != GRB.OPTIMAL:
        return None
    return np.array([w[i].X for i in range(P)])


mu_Yw_check = float(mu_u @ owned_vec - mu_G_r)  # mu_Yw of the manager's *current* squad, just for the branch sign
sense = 1 if mu_Yw_check >= 0 else -1
print(f"current squad's mu_Yw = {mu_Yw_check:.2f} -> using the "
      f"{'mu - lambda*var' if sense == 1 else 'mu + lambda*var'} branch")

gurobi_model, gurobi_w, lin_expr, quad_var_expr, lin_cov_expr = build_mean_variance_model(
    mu_u, Sigma_u, sigma_delta_G_r, price_vec, position_vec, owned_vec, own_budget, own_T_limit
)

current squad's mu_Yw = 8.44 -> using the mu - lambda*var branch
Set parameter Username
Set parameter LicenseID to value 2845129
Academic license - for non-commercial use only - expires 2027-07-16


### Step 6: pick $\lambda^*$ and report $w^*$

For each $\lambda$ in a grid, solve for $w(\lambda)$, then use the _same_ Monte Carlo
$(\delta,G^{(r')})$ samples from Step 4 to estimate $\widehat{\text{Prob}}(Y_{w(\lambda)}>0)
= \frac1{N}\sum_i \mathbb{1}[w(\lambda)^T\delta_i > G^{(r')}_i]$ -- the actual quantity
Eq:maxProb wants maximized. $\lambda^*$ is whichever grid point achieves the highest
empirical probability.


In [12]:
lambda_grid = np.concatenate([[0.0], np.geomspace(0.01, 5.0, 15)])

results = []
for lam in lambda_grid:
    w_lam = solve_for_lambda(gurobi_model, gurobi_w, lin_expr, quad_var_expr, lin_cov_expr, P, lam, sense=sense)
    if w_lam is None:
        continue
    Yw_samples = delta_samples @ w_lam - G_r
    prob_beat = float((Yw_samples > 0).mean())
    mean_score = float(mu_u @ w_lam)
    var_score = float(w_lam @ Sigma_u @ w_lam)
    n_transfers = float(((1 - owned_vec) * w_lam).sum())
    results.append({
        "lambda": lam, "prob_beat_opponent": prob_beat, "mean_score": mean_score,
        "var_score": var_score, "n_transfers": n_transfers, "w": w_lam,
    })

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'w'} for r in results])
print(results_df)

best_idx = results_df['prob_beat_opponent'].idxmax()
lambda_star = results_df.loc[best_idx, 'lambda']
w_star = results[best_idx]['w']
squad_star = [universe[i] for i in range(P) if w_star[i] > 0.5]

print(f"\nlambda* = {lambda_star:.4f}, Prob(beat opponent's {R_PCT:.0%}-percentile squad) = "
      f"{results_df.loc[best_idx, 'prob_beat_opponent']:.1%}")
print(f"transfers made: {results_df.loc[best_idx, 'n_transfers']:.0f} (limit was {own_T_limit})")
print(f"squad value: {sum(price_vec[i] for i in range(P) if w_star[i] > 0.5):.0f} (budget was {own_budget:.0f})")
print(f"w* (element ids): {sorted(squad_star)}")

      lambda  prob_beat_opponent  mean_score   var_score  n_transfers
0   0.000000            0.999000   79.289000  198.362974          4.0
1   0.010000            0.999000   79.289000  198.362974          4.0
2   0.015588            0.999000   79.214000  173.502045          4.0
3   0.024298            0.999000   79.214000  173.502045          4.0
4   0.037875            0.999000   79.214000  173.502045          4.0
5   0.059038            0.999000   79.214000  173.502045          4.0
6   0.092028            0.999333   78.791000  173.211803          4.0
7   0.143450            0.999667   78.312000  176.553888          4.0
8   0.223607            0.999667   76.322000  165.483786          4.0
9   0.348553            1.000000   73.652000  180.672317          4.0
10  0.543316            1.000000   73.652000  180.672317          4.0
11  0.846907            1.000000   73.652000  180.672317          4.0
12  1.320138            0.998000   67.770000  176.844722          4.0
13  2.057799        

### Reading the result

`results_df` shows the mean-variance frontier: as $\lambda$ grows, the optimizer trades
expected score for lower variance, and $\widehat{\text{Prob}}(\text{beat opponent})$ traces
out a curve over that frontier rather than moving monotonically with $\lambda$ (a
higher-variance, higher-mean squad isn't automatically worse at clearing a _fixed_
threshold $G^{(r')}$ -- it depends on where $\mu_{Y_w}$ sits relative to 0). $\lambda^*$
is whichever frontier point actually maximizes that probability, which is the quantity
Eq:maxProb asks for -- not the raw mean-score maximizer ($\lambda=0$) or an arbitrarily
conservative one.

Caveats worth being upfront about:

- $\mu_\delta$ and $\Sigma_\delta$ come from `FPL Data/predictors`' backtested model, so
  this only works for gameweeks that pipeline has already scored (GW6-38 of the
  2024-25 season) -- there's no live "predict a genuinely future gameweek" path here
  yet, same limitation the predictor pipeline itself has.
- $\Sigma_\delta$'s correlation structure comes from historical prediction-residual
  co-movement, not a structural assumption (e.g. same-team clean-sheet correlation) --
  it will pick up real patterns to the extent they were present in GW6-37's residuals,
  but isn't guaranteed to generalize much beyond that window.
- The transfer-limit constraint here only counts _this_ week's swap against the banked
  free-transfer count; it doesn't model taking a deliberate point hit for a swap beyond
  $T'_{\text{own}}$, which a real manager might still rationally do if the expected gain
  outweighs the -4.


### Ground truth: how did $w^*$ actually do?

Everything above is decided _before_ GW38 is played, using only $\mu_\delta$/$\Sigma_\delta$
(model predictions/historical uncertainty) and the opponent's _simulated_ squad
ensemble $\mathbb{W}_o$. Since GW38 has already been played in this dataset, we can now
check against what actually happened: the opponent's real GW38 squad (not a simulated
one), scored on both `xP` (FPL's official expected-points stat -- what the model was
trained to predict) and `total_points` (the real points that actually counted,
including bonus/cards/appearance points that `xP` doesn't capture). The manager's
current (pre-decision) squad is included too, to see whether the optimizer's suggested
transfers would actually have been worth making.


In [13]:
def squad_actuals(squad, gw_players):
    """Sum of actual xP and actual total_points for a squad (list of element ids) at
    the already-played gameweek. Missing players (shouldn't happen for a real squad,
    but guards against a stale universe/price-table mismatch) are excluded and flagged."""
    rows = gw_players.reindex(squad)
    missing = rows.index[rows['xP'].isna()].tolist()
    if missing:
        print(f"  WARNING: {len(missing)} player(s) had no GW{GW} row and were excluded: {missing}")
    rows = rows.dropna(subset=['xP'])
    return float(rows['xP'].sum()), float(rows['total_points'].sum())


opp_actual_row = league_selections[
    (league_selections['team_id'] == OPPONENT_TEAM_ID) & (league_selections['round'] == GW)
].iloc[0]
opp_actual_squad = ast.literal_eval(opp_actual_row['squad'])

own_current_xp, own_current_pts = squad_actuals(sorted(own_current_squad), gw_players)
own_optimal_xp, own_optimal_pts = squad_actuals(squad_star, gw_players)
opp_actual_xp, opp_actual_pts = squad_actuals(opp_actual_squad, gw_players)

comparison_df = pd.DataFrame([
    {"squad": f"own current (GW{GW - 1}, pre-decision)", "actual_xP": own_current_xp, "actual_points": own_current_pts},
    {"squad": f"own optimized w* (lambda*={lambda_star:.4f})", "actual_xP": own_optimal_xp, "actual_points": own_optimal_pts},
    {"squad": f"opponent's real GW{GW} squad", "actual_xP": opp_actual_xp, "actual_points": opp_actual_pts},
])
comparison_df

,squad,actual_xP,actual_points
0,"own current (GW37, pre-decision)",73.7,59.0
1,own optimized w* (lambda*=0.3486),78.1,71.0
2,opponent's real GW38 squad,65.5,50.0


In [14]:
def outcome(a, b):
    return "WIN" if a > b else ("LOSS" if a < b else "TIE")


print(f"xP:     current {own_current_xp:.1f}  ->  optimized {own_optimal_xp:.1f}  "
      f"(delta {own_optimal_xp - own_current_xp:+.1f})  vs opponent {opp_actual_xp:.1f}")
print(f"points: current {own_current_pts:.0f}  ->  optimized {own_optimal_pts:.0f}  "
      f"(delta {own_optimal_pts - own_current_pts:+.0f})  vs opponent {opp_actual_pts:.0f}")
print()
print(f"optimized vs opponent, on xP:     {outcome(own_optimal_xp, opp_actual_xp)}")
print(f"optimized vs opponent, on points: {outcome(own_optimal_pts, opp_actual_pts)}")
print(f"current   vs opponent, on xP:     {outcome(own_current_xp, opp_actual_xp)}")
print(f"current   vs opponent, on points: {outcome(own_current_pts, opp_actual_pts)}")

xP:     current 73.7  ->  optimized 78.1  (delta +4.4)  vs opponent 65.5
points: current 59  ->  optimized 71  (delta +12)  vs opponent 50

optimized vs opponent, on xP:     WIN
optimized vs opponent, on points: WIN
current   vs opponent, on xP:     WIN
current   vs opponent, on points: WIN


## Running across every gameweek

Reproduces the round-robin fixture (manager 205, seeds 61/16, verbatim from
`longitudinal.ipynb`/`copula_estimation.ipynb`) so each week's opponent is known, wraps
the walkthrough above (Steps 1-6 + the ground-truth check) into a single function, and
runs it for every gameweek with enough history on both sides: the opponent's copula fit
(`run_copula_pipeline`'s own `min_history_weeks`) and `mu_delta`/`Sigma_delta`'s
residual history (`FPL Data/predictors` only covers GW6-38, and needs a handful of
rounds _before_ `GW` to estimate a covariance at all).


In [15]:
import random


def create_fixtures(managers, seed=None):
    """Randomized single round-robin fixture (verbatim from longitudinal.ipynb)."""
    if seed is not None:
        random.seed(seed)
    managers = managers.copy()
    random.shuffle(managers)
    fixed = managers[0]
    rotating = managers[1:]
    fixtures = {}
    for gw in range(1, len(managers)):
        current = [fixed] + rotating
        gw_fixtures = {}
        for i in range(len(current) // 2):
            m1, m2 = current[i], current[-(i + 1)]
            gw_fixtures[m1] = m2
            gw_fixtures[m2] = m1
        fixtures[gw] = gw_fixtures
        rotating = [rotating[-1]] + rotating[:-1]
    return fixtures


def get_opponent(manager_id, gameweek, fixtures):
    return fixtures[gameweek][manager_id]


managers_list = [
    205, 901, 4213, 7497, 13603,
    23881, 38054, 38351, 69718, 152245,
    198971, 577273, 593532, 696127, 749244,
    1222335, 1441046, 1546929, 2634457, 4131447,
]
fixtures_leg_1 = create_fixtures(managers_list, seed=61)
fixtures_leg_2 = create_fixtures(managers_list, seed=16)
opponents_by_round = (
    {gw: get_opponent(205, gw, fixtures_leg_1) for gw in range(1, 20)}
    | {gw + 19: get_opponent(205, gw, fixtures_leg_2) for gw in range(1, 20)}
)
print({gw: opponents_by_round[gw] for gw in range(6, 39)})

{6: 23881, 7: 38054, 8: 13603, 9: 593532, 10: 749244, 11: 696127, 12: 4131447, 13: 152245, 14: 1441046, 15: 1222335, 16: 1546929, 17: 198971, 18: 2634457, 19: 577273, 20: 69718, 21: 4213, 22: 1441046, 23: 4131447, 24: 1222335, 25: 152245, 26: 7497, 27: 901, 28: 198971, 29: 749244, 30: 593532, 31: 23881, 32: 1546929, 33: 577273, 34: 2634457, 35: 696127, 36: 38351, 37: 38054, 38: 13603}


### `run_optimization_pipeline`: Steps 1-6 + ground truth, as a function

Returns `None` (with a printed reason) for gameweeks that still can't be run at all --
too little residual history for `Sigma_delta`, no own-squad row, or no feasible
$w(\lambda)$ -- rather than raising, so the bulk loop below can skip them cleanly,
mirroring `copula_estimation.ipynb`'s `run_copula_pipeline`.

**Fallback when $\mathbb{W}_o$ is empty**: if the copula/vine stage can't produce any
feasible opponent squads for that gameweek (either `run_copula_pipeline` itself returns
`None` for want of history, or it runs but nothing survives the transfer/budget filter
-- both happen early in the season), fall back to a single-squad "ensemble": the
opponent's actual squad _entering_ `GW` (their GW-1 squad, before that week's
transfers -- not their real GW outcome, which isn't known yet at decision time and
would leak future information into the optimization). With $O=1$, the order statistic
$G^{(r')}$ collapses to that one squad's score for every $\lambda$, which is a
reasonable degradation: "beat whatever they currently have" is the natural fallback
once there's no distribution of plausible opponent squads to speak of.


In [16]:
def run_optimization_pipeline(GW, OWN_TEAM_ID, OPPONENT_TEAM_ID, R_PCT=0.5,
                               N_SAMPLES_OPP=20_000, N_MC=2000, lambda_grid=None,
                               min_resid_rounds=1, seed=42, verbose=True,
                               own_current_squad_override=None, own_T_limit_override=None):
    if lambda_grid is None:
        lambda_grid = np.concatenate([[0.0], np.geomspace(0.01, 5.0, 10)])

    # --- Step 1: opponent Wo, falling back to their actual GW-1 squad if empty ---
    opp_result = run_copula_pipeline(GW, OPPONENT_TEAM_ID, N_SAMPLES=N_SAMPLES_OPP, seed=seed, verbose=False)
    used_fallback_benchmark = opp_result is None or opp_result['n_unique_feasible_squads'] == 0
    if used_fallback_benchmark:
        opp_prev_rows = league_selections[
            (league_selections['team_id'] == OPPONENT_TEAM_ID) & (league_selections['round'] == GW - 1)
        ]
        if len(opp_prev_rows) == 0:
            if verbose:
                print(f"GW{GW}: opponent Wo empty AND no opponent GW{GW - 1} squad row, skipping")
            return None
        opp_current_squad = tuple(sorted(ast.literal_eval(opp_prev_rows.iloc[0]['squad'])))
        opp_squad_counts = {opp_current_squad: 1}
        if verbose:
            print(f"GW{GW}: opponent Wo empty/unavailable -- falling back to their actual "
                  f"GW{GW - 1} squad as a single-squad benchmark")
    else:
        opp_squad_counts = opp_result['squad_counts']

    # --- Step 2: mu_delta / Sigma_delta, guarding on residual history depth ---
    frames = [pd.read_csv(f'../../predictors/hist/{pos}_preds.csv') for pos in ['gk', 'def', 'mid', 'fwd']]
    all_preds = pd.concat(frames, ignore_index=True)
    n_hist_rounds = all_preds.loc[all_preds['round'] < GW, 'round'].nunique()
    if n_hist_rounds < min_resid_rounds:
        if verbose:
            print(f"GW{GW}: only {n_hist_rounds} residual-history rounds (< {min_resid_rounds}), skipping")
        return None
    delta_elements, mu_delta, Sigma_delta = build_mu_sigma_delta(GW)

    # --- Step 3: own squad, budget, universe ---
    # `own_current_squad_override`/`own_T_limit_override` let a caller (e.g. the rolling
    # season simulation below) inject a *simulated* squad/FT-bank state in place of the
    # real historical one -- everything downstream treats it identically either way.
    if own_current_squad_override is not None:
        own_current_squad = set(own_current_squad_override)
    else:
        own_prev_rows = league_selections[
            (league_selections['team_id'] == OWN_TEAM_ID) & (league_selections['round'] == GW - 1)
        ]
        if len(own_prev_rows) == 0:
            if verbose:
                print(f"GW{GW}: no own GW{GW - 1} squad row, skipping")
            return None
        own_current_squad = set(ast.literal_eval(own_prev_rows.iloc[0]['squad']))

    gw_players_ = player_data[player_data['round'] == GW].drop_duplicates('element').set_index('element')
    universe_ = sorted(set(delta_elements) & set(gw_players_.index))
    if len(universe_) < 15:
        if verbose:
            print(f"GW{GW}: player universe too small ({len(universe_)}), skipping")
        return None

    own_budget_ = float(gw_players_.loc[list(own_current_squad & set(gw_players_.index)), 'value'].sum())
    own_T_limit_ = (own_T_limit_override if own_T_limit_override is not None
                    else compute_free_transfers_available(OWN_TEAM_ID, GW, league_selections))

    price_vec_ = gw_players_.loc[universe_, 'value'].to_numpy(dtype=float)
    position_vec_ = gw_players_.loc[universe_, 'position'].to_numpy()
    owned_vec_ = np.array([1.0 if e in own_current_squad else 0.0 for e in universe_])

    delta_index_ = {e: i for i, e in enumerate(delta_elements)}
    u_idx_ = np.array([delta_index_[e] for e in universe_])
    mu_u_ = mu_delta[u_idx_]
    Sigma_u_ = Sigma_delta[np.ix_(u_idx_, u_idx_)]
    P_ = len(universe_)

    # --- Step 4: Monte Carlo (delta, G^{r'}) ---
    rng_ = np.random.default_rng(seed)
    opp_element_index_ = {e: i for i, e in enumerate(universe_)}
    Wo_rows_, Wo_weights_ = [], []
    for squad, count in opp_squad_counts.items():
        row = np.zeros(P_)
        valid = True
        for e in squad:
            idx = opp_element_index_.get(e)
            if idx is None:
                valid = False
                break
            row[idx] = 1.0
        if valid:
            Wo_rows_.append(row)
            Wo_weights_.append(count)
    if len(Wo_rows_) == 0:
        if verbose:
            print(f"GW{GW}: no opponent squad overlaps with the player universe, skipping")
        return None
    Wo_ = np.array(Wo_rows_)

    r_index_from_top_ = int(np.ceil((1 - R_PCT) * Wo_.shape[0]))
    r_index_from_top_ = min(max(r_index_from_top_, 1), Wo_.shape[0])

    delta_samples_ = rng_.multivariate_normal(mu_u_, Sigma_u_, size=N_MC, method='eigh')
    G_all_ = delta_samples_ @ Wo_.T
    G_sorted_ = np.sort(G_all_, axis=1)
    G_r_ = G_sorted_[:, -r_index_from_top_]
    mu_G_r_ = G_r_.mean()
    sigma_delta_G_r_ = ((delta_samples_ - mu_u_) * (G_r_ - mu_G_r_)[:, None]).mean(axis=0)

    # --- Steps 5-6: mean-variance MIQP over the lambda grid ---
    mu_Yw_check_ = float(mu_u_ @ owned_vec_ - mu_G_r_)
    sense_ = 1 if mu_Yw_check_ >= 0 else -1
    m_, w_, lin_, quad_var_, lin_cov_ = build_mean_variance_model(
        mu_u_, Sigma_u_, sigma_delta_G_r_, price_vec_, position_vec_, owned_vec_, own_budget_, own_T_limit_
    )

    best_prob, best_w, best_lam = -1.0, None, None
    for lam in lambda_grid:
        w_lam = solve_for_lambda(m_, w_, lin_, quad_var_, lin_cov_, P_, lam, sense=sense_)
        if w_lam is None:
            continue
        prob_beat = float((delta_samples_ @ w_lam - G_r_ > 0).mean())
        if prob_beat > best_prob:
            best_prob, best_w, best_lam = prob_beat, w_lam, lam
    if best_w is None:
        if verbose:
            print(f"GW{GW}: no feasible w(lambda) found, skipping")
        return None
    squad_star_ = [universe_[i] for i in range(P_) if best_w[i] > 0.5]

    # --- ground truth ---
    opp_actual_rows = league_selections[
        (league_selections['team_id'] == OPPONENT_TEAM_ID) & (league_selections['round'] == GW)
    ]
    opp_actual_squad_ = ast.literal_eval(opp_actual_rows.iloc[0]['squad']) if len(opp_actual_rows) else None

    def actuals(squad):
        if squad is None:
            return None, None
        rows = gw_players_.reindex(squad).dropna(subset=['xP'])
        return float(rows['xP'].sum()), float(rows['total_points'].sum())

    cur_xp_, cur_pts_ = actuals(sorted(own_current_squad))
    opt_xp_, opt_pts_ = actuals(squad_star_)
    opp_xp_, opp_pts_ = actuals(opp_actual_squad_)

    result = {
        "GW": GW, "own_team_id": OWN_TEAM_ID, "opponent_team_id": OPPONENT_TEAM_ID,
        "lambda_star": best_lam, "prob_beat_opponent": best_prob,
        "used_fallback_benchmark": used_fallback_benchmark,
        "own_T_limit": own_T_limit_, "own_budget": own_budget_,
        "own_current_squad": sorted(own_current_squad), "squad_star": squad_star_,
        "current_xP": cur_xp_, "current_points": cur_pts_,
        "optimized_xP": opt_xp_, "optimized_points": opt_pts_,
        "opponent_xP": opp_xp_, "opponent_points": opp_pts_,
    }
    if verbose:
        def outcome_(a, b):
            return "N/A" if a is None or b is None else ("WIN" if a > b else ("LOSS" if a < b else "TIE"))
        flag = " [fallback benchmark]" if used_fallback_benchmark else ""
        print(f"GW{GW:>2} vs {OPPONENT_TEAM_ID:>8}: lambda*={best_lam:.3f} P(beat)={best_prob:.1%}{flag} | "
              f"xP {cur_xp_:.1f}->{opt_xp_:.1f} vs opp {opp_xp_}  [{outcome_(opt_xp_, opp_xp_)}] | "
              f"pts {cur_pts_:.0f}->{opt_pts_:.0f} vs opp {opp_pts_}  [{outcome_(opt_pts_, opp_pts_)}]")
    return result

### Bulk run

`N_SAMPLES_OPP`/`N_MC`/the $\lambda$ grid are all reduced from the single-GW
walkthrough's defaults to keep 33 gameweeks' worth of copula fits + MIQP solves
tractable in one run -- rerun any individual week with the walkthrough's higher-fidelity
settings if you want a more precise read on it. Results are saved to
`./estimates/optimization_results_all.joblib`, keyed by `GW`.


In [17]:
opt_results_by_gw = {}
for gw in range(6, 39):
    opp_team = opponents_by_round[gw]
    try:
        opt_results_by_gw[gw] = run_optimization_pipeline(
            gw, OWN_TEAM_ID, opp_team, R_PCT=0.5, N_SAMPLES_OPP=20_000, N_MC=2000, seed=42
        )
    except Exception as e:
        print(f"GW{gw} vs {opp_team} FAILED: {e}")
        opt_results_by_gw[gw] = None

joblib.dump(opt_results_by_gw, './estimates/optimization_results_all.joblib')

n_ok = sum(1 for r in opt_results_by_gw.values() if r is not None)
print(f"\n{n_ok} / {len(opt_results_by_gw)} gameweeks produced a result")

GW6: opponent Wo empty/unavailable -- falling back to their actual GW5 squad as a single-squad benchmark
GW6: only 0 residual-history rounds (< 1), skipping
GW 7 vs    38054: lambda*=0.000 P(beat)=100.0% | xP 66.0->66.3 vs opp 63.7  [WIN] | pts 51->35 vs opp 41.0  [LOSS]
GW8: opponent Wo empty/unavailable -- falling back to their actual GW7 squad as a single-squad benchmark
GW 8 vs    13603: lambda*=0.000 P(beat)=100.0% [fallback benchmark] | xP 47.8->55.1 vs opp 47.2  [WIN] | pts 36->29 vs opp 37.0  [LOSS]
GW 9 vs   593532: lambda*=0.000 P(beat)=100.0% | xP 59.0->65.8 vs opp 59.89999999999999  [WIN] | pts 68->61 vs opp 72.0  [LOSS]
GW10 vs   749244: lambda*=0.000 P(beat)=100.0% | xP 55.6->63.2 vs opp 55.0  [WIN] | pts 57->46 vs opp 48.0  [LOSS]
GW11 vs   696127: lambda*=0.000 P(beat)=100.0% | xP 61.8->75.9 vs opp 57.0  [WIN] | pts 49->57 vs opp 64.0  [LOSS]

Interrupt request received

Interrupt request received


KeyboardInterrupt: 

In [ ]:
summary_rows = []
for gw, res in opt_results_by_gw.items():
    if res is None:
        summary_rows.append({"GW": gw, "opponent": opponents_by_round[gw], "status": "skipped"})
    else:
        summary_rows.append({
            "GW": gw, "opponent": res["opponent_team_id"], "status": "ok",
            "fallback_benchmark": res["used_fallback_benchmark"],
            "lambda_star": res["lambda_star"], "prob_beat_opponent": res["prob_beat_opponent"],
            "current_xP": res["current_xP"], "optimized_xP": res["optimized_xP"], "opponent_xP": res["opponent_xP"],
            "current_points": res["current_points"], "optimized_points": res["optimized_points"],
            "opponent_points": res["opponent_points"],
        })

opt_summary_df = pd.DataFrame(summary_rows)
ok = opt_summary_df[opt_summary_df['status'] == 'ok'].copy()

if len(ok) == 0:
    print(f"0 / {len(opt_summary_df)} gameweeks produced a result -- nothing to summarize "
          f"(see the per-GW skip reasons printed by the bulk-run cell above)")
else:
    ok['win_points'] = ok['optimized_points'] > ok['opponent_points']
    ok['win_xP'] = ok['optimized_xP'] > ok['opponent_xP']
    ok['improved_vs_current_points'] = ok['optimized_points'] > ok['current_points']

    print(f"{len(ok)} / {len(opt_summary_df)} gameweeks with a result "
          f"({ok['fallback_benchmark'].sum()} used the fallback single-squad benchmark)")
    print(f"win rate on actual points, optimized vs opponent: {ok['win_points'].mean():.1%}")
    print(f"win rate on actual xP, optimized vs opponent: {ok['win_xP'].mean():.1%}")
    print(f"optimized squad scored more actual points than staying put: {ok['improved_vs_current_points'].mean():.1%} of weeks")
    print(f"mean points delta (optimized - current): {(ok['optimized_points'] - ok['current_points']).mean():+.1f}")

opt_summary_df

## Rolling season simulation: "what if we'd actually followed this strategy?"

The bulk run above re-optimizes each gameweek independently, always starting from
manager 205's _real_ historical squad at `GW - 1` -- it answers "given what the manager
actually owned that week, what should they have done?" for every week separately, but
each week's answer is thrown away before the next.

A different, path-dependent question: starting from the real squad entering GW6, if the
manager had _actually made every optimizer-recommended transfer_ and never deviated,
what would their squad -- and FT bank -- have looked like by GW38? This carries the
optimizer's own chosen squad (`squad_star`) forward as next week's "current squad"
instead of re-reading real history, and evolves the free-transfer bank under the
_simulated_ transfers actually made each week (same accrual rule as
`compute_free_transfers_available`: capped at 5, `+1` per week not fully spent). A
gameweek `run_optimization_pipeline` can't solve at all (missing residual history, no
opponent data, etc.) leaves the simulated squad unchanged and still banks a transfer,
mirroring a real blank gameweek.

`own_current_squad_override`/`own_T_limit_override` (added to `run_optimization_pipeline`
above) are exactly what make this possible: everything downstream of Step 3 -- budget,
Monte Carlo, the MIQP, ground truth -- is computed off whatever squad/bank state is
passed in, real or simulated, with no other code path changes needed.


In [18]:
def run_rolling_optimization(start_gw, end_gw, OWN_TEAM_ID, R_PCT=0.5,
                              N_SAMPLES_OPP=20_000, N_MC=2000, lambda_grid=None,
                              min_resid_rounds=1, seed=42, verbose=True):
    own_prev_rows = league_selections[
        (league_selections['team_id'] == OWN_TEAM_ID) & (league_selections['round'] == start_gw - 1)
    ]
    if len(own_prev_rows) == 0:
        raise ValueError(f"no real own squad row at GW{start_gw - 1} to seed the rollout from")
    current_squad = set(ast.literal_eval(own_prev_rows.iloc[0]['squad']))
    ft_bank = compute_free_transfers_available(OWN_TEAM_ID, start_gw, league_selections)

    rollout = []
    for gw in range(start_gw, end_gw + 1):
        opp_team = opponents_by_round[gw]
        try:
            res = run_optimization_pipeline(
                gw, OWN_TEAM_ID, opp_team, R_PCT=R_PCT,
                N_SAMPLES_OPP=N_SAMPLES_OPP, N_MC=N_MC, lambda_grid=lambda_grid,
                min_resid_rounds=min_resid_rounds, seed=seed, verbose=verbose,
                own_current_squad_override=current_squad, own_T_limit_override=ft_bank,
            )
        except Exception as e:
            if verbose:
                print(f"GW{gw} vs {opp_team} FAILED: {e}")
            res = None

        if res is None:
            # couldn't solve this gameweek at all -- carry the simulated squad forward
            # unchanged, same as a real week with no transfer made.
            ft_bank = min(5, ft_bank + 1)
            rollout.append({
                "GW": gw, "opponent": opp_team, "status": "skipped",
                "n_transfers": 0, "ft_bank_after": ft_bank, "squad": sorted(current_squad),
            })
            continue

        new_squad = set(res["squad_star"])
        n_transfers = len(new_squad - current_squad)
        ft_bank = min(5, (ft_bank - n_transfers) + 1)
        current_squad = new_squad

        rollout.append({
            "GW": gw, "opponent": opp_team, "status": "ok",
            "used_fallback_benchmark": res["used_fallback_benchmark"],
            "n_transfers": n_transfers, "ft_bank_after": ft_bank, "squad": sorted(current_squad),
            "sim_current_xP": res["current_xP"], "sim_optimized_xP": res["optimized_xP"], "opponent_xP": res["opponent_xP"],
            "sim_current_points": res["current_points"], "sim_optimized_points": res["optimized_points"],
            "opponent_points": res["opponent_points"],
        })

    return pd.DataFrame(rollout)


ROLLING_START_GW = 6
ROLLING_END_GW = 38

rolling_df = run_rolling_optimization(ROLLING_START_GW, ROLLING_END_GW, OWN_TEAM_ID,
                                       R_PCT=0.5, N_SAMPLES_OPP=20_000, N_MC=2000, seed=42)
joblib.dump(rolling_df, './estimates/rolling_optimization_results.joblib')

n_ok_roll = (rolling_df['status'] == 'ok').sum()
print(f"\n{n_ok_roll} / {len(rolling_df)} gameweeks produced a rolling result")
rolling_df

GW6: opponent Wo empty/unavailable -- falling back to their actual GW5 squad as a single-squad benchmark
GW6: only 0 residual-history rounds (< 1), skipping
GW 7 vs    38054: lambda*=0.000 P(beat)=100.0% | xP 53.5->56.5 vs opp 63.7  [LOSS] | pts 39->29 vs opp 41.0  [LOSS]
GW8: opponent Wo empty/unavailable -- falling back to their actual GW7 squad as a single-squad benchmark
GW 8 vs    13603: lambda*=0.000 P(beat)=0.0% [fallback benchmark] | xP 56.6->61.9 vs opp 47.2  [WIN] | pts 59->60 vs opp 37.0  [WIN]
GW 9 vs   593532: lambda*=0.079 P(beat)=93.0% | xP 59.8->62.9 vs opp 59.89999999999999  [WIN] | pts 55->59 vs opp 72.0  [LOSS]
GW10 vs   749244: lambda*=0.000 P(beat)=83.5% | xP 58.6->59.4 vs opp 55.0  [WIN] | pts 56->56 vs opp 48.0  [WIN]
GW11 vs   696127: lambda*=0.158 P(beat)=87.1% | xP 63.6->65.0 vs opp 57.0  [WIN] | pts 57->53 vs opp 64.0  [LOSS]
GW12 vs  4131447: lambda*=0.000 P(beat)=50.9% | xP 47.7->49.1 vs opp 65.3  [LOSS] | pts 40->33 vs opp 61.0  [LOSS]
GW13 vs   152245: la

,GW,opponent,status,n_transfers,ft_bank_after,squad,used_fallback_benchmark,sim_current_xP,sim_optimized_xP,opponent_xP,sim_current_points,sim_optimized_points,opponent_points
0,6,23881,skipped,0,2,"[19, 54, 58, 201, 209, 231, 251, 255, 311, 317...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7,38054,ok,2,1,"[19, 54, 58, 180, 201, 209, 231, 255, 311, 328...",False,53.5,56.5,63.7,39.0,29.0,41.0
2,8,13603,ok,1,1,"[54, 58, 180, 201, 209, 231, 255, 311, 328, 35...",True,56.6,61.9,47.2,59.0,60.0,37.0
3,9,593532,ok,1,1,"[54, 58, 201, 209, 231, 255, 311, 328, 350, 35...",False,59.8,62.9,59.9,55.0,59.0,72.0
4,10,749244,ok,1,1,"[54, 110, 201, 209, 231, 255, 311, 328, 350, 3...",False,58.6,59.4,55.0,56.0,56.0,48.0
5,11,696127,ok,1,1,"[54, 110, 201, 209, 255, 311, 328, 350, 351, 3...",False,63.6,65.0,57.0,57.0,53.0,64.0
6,12,4131447,ok,1,1,"[54, 94, 110, 201, 209, 255, 311, 328, 350, 35...",False,47.7,49.1,65.3,40.0,33.0,61.0
7,13,152245,ok,1,1,"[89, 94, 110, 201, 209, 255, 311, 328, 350, 35...",False,50.2,52.0,71.5,51.0,54.0,67.0
8,14,1441046,ok,1,1,"[71, 89, 94, 110, 201, 209, 255, 311, 328, 350...",True,55.1,62.6,86.9,65.0,68.0,56.0
9,15,1222335,skipped,0,2,"[71, 89, 94, 110, 201, 209, 255, 311, 328, 350...",NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Season-long comparison: rolling-optimized vs. what actually happened

`rolling_df`'s `sim_optimized_points` is the simulated trajectory's actual points each
week (the squad _that trajectory_ held going into the gameweek). This pulls the
manager's _real_ historical squad's actual points and the opponent's real actual points
for every gameweek directly from `league_selections` (independent of whether a given
week could be solved), so all three trajectories -- rolling-simulated, real history, and
the real opponents -- are compared on the same actually-played outcomes.


In [19]:
def real_squad_actuals(team_id, gw):
    rows = league_selections[(league_selections['team_id'] == team_id) & (league_selections['round'] == gw)]
    if len(rows) == 0:
        return None, None
    squad = ast.literal_eval(rows.iloc[0]['squad'])
    gw_players_ = player_data[player_data['round'] == gw].drop_duplicates('element').set_index('element')
    rows_p = gw_players_.reindex(squad).dropna(subset=['xP'])
    return float(rows_p['xP'].sum()), float(rows_p['total_points'].sum())


compare_rows = []
for gw in range(ROLLING_START_GW, ROLLING_END_GW + 1):
    real_xp, real_pts = real_squad_actuals(OWN_TEAM_ID, gw)
    opp_xp, opp_pts = real_squad_actuals(opponents_by_round[gw], gw)
    roll_row = rolling_df.loc[rolling_df['GW'] == gw].iloc[0]
    sim_pts = roll_row.get('sim_optimized_points') if roll_row['status'] == 'ok' else None
    sim_xp = roll_row.get('sim_optimized_xP') if roll_row['status'] == 'ok' else None
    compare_rows.append({
        "GW": gw, "opponent": opponents_by_round[gw], "status": roll_row['status'],
        "rolling_points": sim_pts, "real_points": real_pts, "opponent_points": opp_pts,
        "rolling_xP": sim_xp, "real_xP": real_xp, "opponent_xP": opp_xp,
    })
compare_df = pd.DataFrame(compare_rows)

# Weeks the rollout couldn't solve have no simulated squad of their own for that GW
# (the trajectory just carries the prior squad forward), so `rolling_points` is NaN
# there rather than a fabricated value -- excluded from the cumulative sums below via
# `status`, and flagged in `compare_df` so it's visible which weeks are missing.
solved = compare_df[compare_df['status'] == 'ok'].copy()
print(f"{len(solved)} / {len(compare_df)} rolling gameweeks solved and included in the cumulative comparison")

solved['cum_rolling_points'] = solved['rolling_points'].cumsum()
solved['cum_real_points'] = solved['real_points'].cumsum()
solved['cum_opponent_points'] = solved['opponent_points'].cumsum()

print(f"\ncumulative points over solved gameweeks ({ROLLING_START_GW}-{ROLLING_END_GW}):")
print(f"  rolling-optimized trajectory: {solved['cum_rolling_points'].iloc[-1]:.0f}")
print(f"  real historical trajectory:   {solved['cum_real_points'].iloc[-1]:.0f}")
print(f"  opponents (real, that week):  {solved['cum_opponent_points'].iloc[-1]:.0f}")
print(f"\nrolling beat real history in {(solved['rolling_points'] > solved['real_points']).mean():.1%} of solved weeks")
print(f"rolling beat that week's opponent in {(solved['rolling_points'] > solved['opponent_points']).mean():.1%} of solved weeks")

compare_df

29 / 33 rolling gameweeks solved and included in the cumulative comparison

cumulative points over solved gameweeks (6-38):
  rolling-optimized trajectory: 1559
  real historical trajectory:   1765
  opponents (real, that week):  1766

rolling beat real history in 37.9% of solved weeks
rolling beat that week's opponent in 34.5% of solved weeks


,GW,opponent,status,rolling_points,real_points,opponent_points,rolling_xP,real_xP,opponent_xP
0,6,23881,skipped,NaN,50.0,45.0,NaN,72.5,72.5
1,7,38054,ok,29.0,51.0,41.0,56.5,66.0,63.7
2,8,13603,ok,60.0,40.0,37.0,61.9,55.5,47.2
3,9,593532,ok,59.0,68.0,72.0,62.9,59.0,59.9
4,10,749244,ok,56.0,58.0,48.0,59.4,57.6,55.0
5,11,696127,ok,53.0,49.0,64.0,65.0,61.8,57.0
6,12,4131447,ok,33.0,55.0,61.0,49.1,55.1,65.3
7,13,152245,ok,54.0,82.0,67.0,52.0,66.0,71.5
8,14,1441046,ok,68.0,51.0,56.0,62.6,65.2,86.9
9,15,1222335,skipped,NaN,55.0,57.0,NaN,54.9,65.4


In [24]:
(compare_df['real_xP'] < compare_df['opponent_xP']).sum()

14